# WER on `train_all`

Corpus and per-region WER for the dialect-aware (FHNW STT4SG) and dialect-ignorant (Whisper-large-v2) hypotheses on the `train_all` split.

In [1]:
from pathlib import Path

import pandas as pd
from jiwer import wer

# locate project root by walking up until requirements.txt is found
_anchor = Path.cwd().resolve()
PROJECT_ROOT = next(p for p in (_anchor, *_anchor.parents) if (p / "requirements.txt").exists())

DAT_TSV = PROJECT_ROOT / "transcripts" / "dialect-aware" / "fhnw" / "stt4sg" / "train_all_transcribed.tsv"
DIT_TSV = PROJECT_ROOT / "transcripts" / "dialect-ignorant" / "whisper-large-v2" / "stt4sg" / "train_all_enriched_transcribed_praet.tsv"

In [2]:
dat = pd.read_csv(
    DAT_TSV, sep="\t", encoding="utf-8-sig",
    usecols=["path", "dialect_region", "sentence", "fhnw_transcript"],
)
dit = pd.read_csv(
    DIT_TSV, sep="\t", encoding="utf-8-sig",
    usecols=["path", "clip_is_usable", "whisper_large_v2_transcript"],
)

# `clip_is_usable` may be stored as the strings "True"/"False"; .astype(bool) would treat
# any non-empty string as truthy, so normalize explicitly before filtering.
dit = dit[dit["clip_is_usable"].astype(str).str.strip().str.lower().eq("true")]

dat = dat[~dat["fhnw_transcript"].fillna("").astype(str).str.startswith("ERROR")]
dit = dit[~dit["whisper_large_v2_transcript"].fillna("").astype(str).str.startswith("ERROR")]

df = dat.merge(dit.drop(columns="clip_is_usable"), on="path", how="inner")
df = df[
    df["sentence"].fillna("").str.strip().ne("")
    & df["fhnw_transcript"].fillna("").str.strip().ne("")
    & df["whisper_large_v2_transcript"].fillna("").str.strip().ne("")
].reset_index(drop=True)

print(f"Evaluated clips: {len(df):,}")
print(f"Regions:         {df['dialect_region'].nunique()}")

Evaluated clips: 198,274
Regions:         7


In [3]:
refs = df["sentence"].tolist()
dat_wer = wer(refs, df["fhnw_transcript"].tolist())
dit_wer = wer(refs, df["whisper_large_v2_transcript"].tolist())

print(f"DAT (FHNW) WER: {dat_wer:.4f}")
print(f"DIT (Whisper-large-v2) WER: {dit_wer:.4f}")
print(f"Delta (DIT − DAT): {dit_wer - dat_wer:+.4f}")

DAT (FHNW) WER: 0.1416
DIT (Whisper-large-v2) WER: 0.2686
Delta (DIT − DAT): +0.1270


In [4]:
REGION_ORDER = ["Wallis", "Zürich", "Bern", "Basel", "Graubünden", "Innerschweiz", "Ostschweiz"]

rows = []
for region in REGION_ORDER:
    sub = df[df["dialect_region"] == region]
    if sub.empty:
        continue
    rows.append({
        "region": region,
        "n": len(sub),
        "DAT WER": wer(sub["sentence"].tolist(), sub["fhnw_transcript"].tolist()),
        "DIT WER": wer(sub["sentence"].tolist(), sub["whisper_large_v2_transcript"].tolist()),
    })
per_region = pd.DataFrame(rows)
per_region["Δ (DIT − DAT)"] = per_region["DIT WER"] - per_region["DAT WER"]
per_region

,region,n,DAT WER,DIT WER,Δ (DIT − DAT)
0,Wallis,29426,0.165251,0.374538,0.209288
1,Zürich,28711,0.135940,0.239090,0.103150
2,Bern,28514,0.138817,0.251812,0.112995
3,Basel,27177,0.129679,0.245106,0.115427
4,Graubünden,23939,0.146183,0.262608,0.116425
5,Innerschweiz,29352,0.130305,0.226229,0.095923
6,Ostschweiz,31155,0.144498,0.275868,0.131370
